<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 02. Árboles de Regresión y Poda (Cost-Complexity)
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 09
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/09%20-%20Decision%20Trees/Para%20Dummies/02_Arboles_Regresion_y_Poda_Cost_Complexity_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 📈

Hasta ahora el árbol solo respondía preguntas de "Sí" o "No" (clasificación). Pero, ¿qué pasa si lo que queremos predecir es un número, como el precio de una casa o la temperatura de mañana?

Este cuaderno es la versión sencilla del módulo 02 (Árboles de Regresión y Poda). Aquí verás, con analogías y ejemplos pequeños:

1. Cómo un árbol puede predecir números en lugar de categorías.
2. Por qué un árbol demasiado grande memoriza los datos en vez de aprender de ellos (sobreajuste).
3. Qué es "podar" un árbol y por qué eso lo hace más útil en la vida real.

---
## 1. Predecir un número a punta de preguntas: escaleras, no rampas 🪜

Un árbol de regresión sigue funcionando con preguntas de Sí/No, pero en vez de terminar en una categoría, cada hoja termina en un **número**: el promedio de todos los ejemplos que cayeron ahí.

Piensa en una escalera en lugar de una rampa: una rampa sube suavemente, sin escalones. Un árbol de regresión, en cambio, predice "por escalones": todos los datos que responden igual a las mismas preguntas reciben exactamente la misma predicción (el promedio de su grupo), aunque en la realidad los valores varíen un poco.

Vamos a comprobarlo generando datos con una forma suave (una curva de seno) y viendo cómo el árbol la aproxima a punta de escalones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

np.random.seed(42)
X = np.sort(np.random.uniform(-3, 3, 60)).reshape(-1, 1)
y = np.sin(X).flatten() + np.random.normal(0, 0.15, 60)

arbol_pequeno = DecisionTreeRegressor(max_depth=2, random_state=42).fit(X, y)

x_linea = np.linspace(-3, 3, 300).reshape(-1, 1)
plt.figure(figsize=(8, 4))
plt.scatter(X, y, color='black', alpha=0.6, label='Datos reales')
plt.plot(x_linea, arbol_pequeno.predict(x_linea), color='#2563eb', lw=2.5, label='Predicción del árbol (max_depth=2)')
plt.title('Un árbol de regresión predice "por escalones"', fontweight='bold')
plt.legend()
plt.show()

### 🤔 ¿Qué acaba de pasar?

- Creamos datos que siguen (aproximadamente) una curva suave: `np.sin(X)` más un poco de ruido aleatorio.
- `DecisionTreeRegressor(max_depth=2)` solo puede hacer 2 niveles de preguntas, así que divide el eje X en, como máximo, 4 tramos.
- Dentro de cada tramo, la predicción es **plana** (un escalón): es el promedio de los datos de entrenamiento que cayeron ahí.
- Por eso la línea azul se ve como una escalera y no como una curva suave: así predicen siempre los árboles de regresión, sin importar cuántas preguntas hagas — solo se vuelven escalones más angostos.

---
## 2. El peligro de un árbol sin límites: memorizar en vez de aprender ✂️

Si dejamos que el árbol crezca sin ningún freno, seguirá haciendo preguntas hasta que cada hoja tenga un solo dato. El resultado: acierta el 100% en los datos que ya vio (los memorizó), pero falla mucho con datos nuevos. A esto se le llama **sobreajuste (overfitting)**.

Es como un estudiante que memoriza las respuestas exactas del examen de práctica, palabra por palabra, en vez de entender el tema: le va perfecto si repiten el mismo examen, pero fracasa si cambian una sola pregunta.

La solución es **podar** el árbol, igual que se podan las ramas débiles de un árbol de jardín para que crezca sano: eliminar las divisiones que no aportan mucho, aunque eso signifique "perder" algo de exactitud en los datos de entrenamiento. En Scikit-Learn, el parámetro que controla la poda se llama `ccp_alpha`: entre más alto, más ramas se recortan.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

arbol_sin_freno = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)
arbol_podado = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train, y_train)

print('Árbol SIN freno (crece libre):')
print(f'  R² en entrenamiento: {arbol_sin_freno.score(X_train, y_train):.3f}')
print(f'  R² en prueba:        {arbol_sin_freno.score(X_test, y_test):.3f}')

print('\nÁrbol PODADO (max_depth=3):')
print(f'  R² en entrenamiento: {arbol_podado.score(X_train, y_train):.3f}')
print(f'  R² en prueba:        {arbol_podado.score(X_test, y_test):.3f}')

### 🤔 ¿Qué acaba de pasar?

- `train_test_split` separa los datos en dos grupos: uno para entrenar (que el árbol sí ve) y otro para probar (que el árbol nunca ve durante el entrenamiento).
- El árbol sin freno consigue un R² muy alto (cercano a 1.0) en entrenamiento — casi memoriza cada punto — pero su R² en prueba suele ser notablemente más bajo.
- El árbol podado (`max_depth=3`) tiene un R² de entrenamiento más modesto, pero su R² en prueba suele acercarse más al de entrenamiento: eso significa que generaliza mejor a datos que no ha visto.
- R² (coeficiente de determinación) va de 0 a 1 (puede ser negativo si el modelo es malísimo): entre más cerca de 1, mejor explica el modelo la variación de los datos.

---
##### 🎯 Reto Práctico para Dummies: Encuentra el punto medio

Prueba entrenar árboles con `max_depth` igual a 1, 3, 5 y `None` (sin límite) sobre los mismos `X_train`, `y_train`, y compara el R² en prueba de cada uno. ¿Cuál profundidad te parece la más equilibrada?

In [ ]:
# =========================================================================
# TU SOLUCIÓN: Reto Dummies 2 - Probando distintas profundidades
# =========================================================================

# for profundidad in [1, 3, 5, None]:
#     modelo = DecisionTreeRegressor(max_depth=profundidad, random_state=42).fit(X_train, y_train)
#     print(...)


<details>
<summary><b>💡 Haz clic aquí para ver la solución explicada...</b></summary>

```python
for profundidad in [1, 3, 5, None]:
    modelo = DecisionTreeRegressor(max_depth=profundidad, random_state=42).fit(X_train, y_train)
    r2_train = modelo.score(X_train, y_train)
    r2_test = modelo.score(X_test, y_test)
    print(f'max_depth={str(profundidad):>4} -> R² train: {r2_train:.3f} | R² test: {r2_test:.3f}')

print('👉 Fíjate en qué profundidad el R² de prueba deja de mejorar (o empieza a empeorar): ahí es donde comienza el sobreajuste.')
```
</details>

---
## 3. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| Árbol de regresión | Predice números (no categorías); cada hoja da el promedio de los datos que cayeron en ella. |
| Predicción "por escalones" | El árbol no dibuja curvas suaves, sino tramos planos separados por las preguntas que hizo. |
| Sobreajuste (overfitting) | Cuando el árbol memoriza los datos de entrenamiento y falla con datos nuevos. |
| Poda (pruning) | Recortar ramas del árbol para que generalice mejor, aunque "pierda" algo de exactitud en entrenamiento. |
| `ccp_alpha` / `max_depth` | Parámetros que controlan qué tanto se poda o limita el crecimiento del árbol. |

➡️ **Siguiente paso:** en el cuaderno [03 - Métodos de Ensamble: Bagging y Random Forests (Para Dummies)](03_Metodos_Ensamble_Bagging_y_Random_Forests_Dummies.ipynb) veremos qué pasa cuando, en vez de un solo árbol, combinamos las opiniones de muchos árboles a la vez.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
